# Wine Quality Dataset

In [1]:
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random
from sklearn.model_selection import StratifiedKFold, learning_curve, cross_validate
from sklearn.metrics import confusion_matrix

SEED = 49
np.random.seed(SEED)
random.seed(SEED)

### Data Loading

We load the Wine Quality dataset and inspect its basic structure.

In [3]:
DATA_PATH = "../wine.csv"
df = pd.read_csv(DATA_PATH)

df.head()

,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol,class,type,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,0,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,0,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,0,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,0,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,0,5


The variables quality, class, and type are categorical in nature, despite being encoded as integers. Specifically, quality is an ordinal multi-class target variable, type is a binary categorical feature, and class represents a discrete categorical grouping. All remaining features are continuous numerical physicochemical measurements.

### Target Variable Distribution

We analyze class balance to motivate metric selection.

In [4]:
target_col = "quality"
df[target_col].value_counts()

4    2251
5    1561
3    1467
6     813
7     204
2     163
1      20
8      18
Name: quality, dtype: int64

In [5]:
df[target_col].value_counts(normalize=True)

4    0.346468
5    0.240265
3    0.225797
6    0.125135
7    0.031399
2    0.025089
1    0.003078
8    0.002771
Name: quality, dtype: float64

Because the Wine dataset is unbalanced, accuracy may overestimate performance in the majority classes. Therefore, we adopt Macro-F1 as the primary evaluation metric.

## Experiments